In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)

BASE_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "category_code"]
BASE = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], codes]).astype(np.float64)
WITH_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "structural", "category_code"]
WITH = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float64)

masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

# ---------------------------------------------------------------------------
# Признаки по мотивам решения, занявшего 4 место в E-CUP 2024 (репозиторий автора
# открыт). Три идеи, которых у нас нет:
#   1. отдельный столбец на каждую важную характеристику категории, а не агрегат;
#   2. мягкое сравнение значений: пересечение токенов, а не точное равенство;
#   3. знаменатель только по общим ключам — отсутствие атрибута не улика.
# ---------------------------------------------------------------------------
import json as _json, re as _re, collections
TOKENS = _re.compile(r"[a-zа-яё0-9]+")
def parse_attrs(raw):
    try: d = _json.loads(raw) if raw else {}
    except Exception: return {}
    return {str(k).strip().lower(): str(v).strip().lower() for k, v in d.items() if str(v).strip()}

ATTRS = {int(i): parse_attrs(a) for i, a in zip(items.id, items.attributes)}
a1, a2 = pairs.id1.to_numpy(), pairs.id2.to_numpy()
item_cat = dict(zip(items.id, items.category.astype(str)))

# Важность характеристики внутри категории: доля карточек, где ключ встречается, взвешенная
# на разнообразие его значений. Ключ, который есть у всех и всегда одинаков, бесполезен.
TOP_N = 12
top_keys = {}
for c in known:
    ids = [int(i) for i in items.id[items.category.astype(str) == c]]
    freq, uniq = collections.Counter(), collections.defaultdict(set)
    for i in ids:
        for k, v in ATTRS[i].items():
            freq[k] += 1; uniq[k].add(v)
    scored = [(freq[k] / len(ids) * min(1.0, len(uniq[k]) / 50), k) for k in freq if freq[k] >= 20]
    top_keys[c] = [k for _, k in sorted(scored, reverse=True)[:TOP_N]]
log("топ-характеристики посчитаны, пример: " + str(top_keys[known[0]][:5]))

def soft(u, v):
    a, b = set(TOKENS.findall(u)), set(TOKENS.findall(v))
    return len(a & b) / len(a | b) if (a | b) else 0.0

def rival_features(x, z):
    A, B = ATTRS.get(int(x), {}), ATTRS.get(int(z), {})
    shared = A.keys() & B.keys()
    exact = sum(1 for k in shared if A[k] == B[k])
    weight = sum(soft(A[k], B[k]) for k in shared)
    union_keys = len(A.keys() | B.keys())
    iou_attr = exact / len(shared) if shared else 0.0
    iou_weight = weight / len(shared) if shared else 0.0
    iou_dict = len(shared) / union_keys if union_keys else 0.0
    vec = [iou_attr, iou_weight, iou_dict, iou_attr * iou_dict, float(len(shared))]
    keys = top_keys.get(item_cat.get(int(x), "?"), [])
    for k in keys:
        if k in A and k in B: vec.append(1.0 if A[k] == B[k] else -1.0)
        elif k in A or k in B: vec.append(0.5)
        else: vec.append(0.0)
    vec += [0.0] * (TOP_N - len(keys))
    return vec

t = time.perf_counter()
RIVAL = np.array([rival_features(x, z) for x, z in zip(a1, a2)], dtype=np.float32)
log(f"признаки соперника посчитаны: {RIVAL.shape} за {time.perf_counter()-t:.0f}с")

codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)
BASE = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float32)
WITH = np.column_stack([BASE, RIVAL]).astype(np.float32)
PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
for tag, mat in [("как сейчас", BASE), ("плюс признаки соперника", WITH)]:
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        p[te] = HistGradientBoostingClassifier(**PARAMS).fit(mat[tr], y[tr]).predict_proba(mat[te])[:, 1]
    mu, sd = macro(rk(p))
    log(f"  {tag:<26} {mu:.6f} ± {sd:.6f}  ({mat.shape[1]} столбцов)")
